# Laboratorium terbuka: pembentukan pola

Notebook ini merupakan pendamping komputasi mandiri untuk Bab 10. Seluruh perhitungan memakai NumPy, SciPy, dan Matplotlib, dapat dijalankan secara luring, serta tidak membaca berkas data eksternal, berkas pengguna, ataupun jaringan. Notebook ini bukan port, tiruan, atau rekonstruksi GUI MATLAB *Patterns*; kode, parameter, kisi, langkah waktu, dan kondisi awal yang menghasilkan Gambar 10.2 tidak tersedia dalam unit sumber beku.

Bagian pertama memeriksa relasi dispersi kompleks persamaan Swift–Hohenberg, menggambar dua kurva pertumbuhan representatif tanpa mengklaim kesamaan piksel dengan Gambar 10.3, dan menutup cabang keberadaan pada Soal 1–3. Bagian kedua menjalankan simulasi spektral dua dimensi berkondisi batas periodik dengan biji tetap, memeriksa pertumbuhan awal modus kritis, skala panjang dominan, determinisme, dan penghalusan langkah waktu. Bagian terakhir merekonstruksi pendahulu dimensional model Klausmeier yang dirujuk bab, lalu memeriksa penskalaan, titik tetap, Jacobian, dan relasi dispersi spasialnya.

**ID unit:** O005-LEGA-V101-CH10  
**ID notebook:** O005-LEGA-V101-CH10-NB01  
**Sumber:** Joceline Lega, *Introduction to Mathematical Modeling*, v1.01 (Maret 2026), Bab 10, ‘Pattern Formation’, University of Arizona Pressbooks: https://opentextbooks.library.arizona.edu/mathematicalmodeling/chapter/pattern-formation/  
**Lisensi:** [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)  
**Pemberitahuan perubahan:** pendamping komputasi Bahasa Indonesia ini baru dan independen; ia menerjemahkan konteks bab serta mengimplementasikan ulang pemeriksaan matematis dengan Python terbuka. Ia bukan terbitan atau dukungan resmi Joceline Lega maupun University of Arizona.  
**Provenans:** persamaan Swift–Hohenberg diturunkan dari Bab 10. Persamaan dimensional Klausmeier ditranskripsikan dan dinyatakan ulang dari Persamaan (1) dalam artikel primer yang dikutip, DOI 10.1126/science.284.5421.1826; tidak ada berkas artikel, kode MATLAB, atau perangkat lunak berpemilik yang disalin atau dibundel.

## Batas model dan pilihan yang dinyatakan

Dengan \(k^2=k_x^2+k_y^2\), simbol Fourier dari linearisasi persamaan Swift–Hohenberg adalah

\[
\lambda(k)=\mu+i\nu-(\alpha+i\beta)(\Omega-k^2)^2-i\eta k^2,
\qquad
\sigma(k)=\operatorname{Re}\lambda(k)=\mu-\alpha(\Omega-k^2)^2.
\]

Analisis kontinu memberi \(k_c=\sqrt{\Omega}\) dan \(\mu_c=0\) jika \(\Omega>0\). Pada kotak periodik hingga, hanya vektor gelombang diskret yang tersedia; ambang sebenarnya menjadi \(\mu_{c,\mathrm{kotak}}=\min_{\mathbf k}\alpha(\Omega-|\mathbf k|^2)^2\). Notebook memeriksa kedua pernyataan ini secara terpisah.

Simulasi menggunakan kotak \(16\pi\times16\pi\), kisi \(64\times64\), integrator spektral ETD orde satu, kondisi awal garis kosinus yang diberi gangguan acak kecil, dan biji generator yang dinyatakan. Pilihan ini sengaja sederhana dan dapat direproduksi; hasilnya merupakan demonstrasi mandiri, bukan reproduksi Gambar 10.2.

In [ ]:
import platform
import time

import numpy as np
import scipy
from scipy.linalg import eigvals
import matplotlib
import matplotlib.pyplot as plt

np.set_printoptions(precision=10, suppress=True)

VERSI_REFERENSI = {
    "Python": "3.13.9",
    "NumPy": "2.4.4",
    "SciPy": "1.17.1",
    "Matplotlib": "3.10.9",
}
versi_aktual = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "SciPy": scipy.__version__,
    "Matplotlib": matplotlib.__version__,
}
assert versi_aktual == VERSI_REFERENSI, (versi_aktual, VERSI_REFERENSI)

def akhiri_gambar(fig):
    fig.canvas.draw()
    assert len(fig.axes) > 0
    if matplotlib.get_backend().lower() == "agg":
        plt.close(fig)
    else:
        plt.show()

assert isinstance(matplotlib.get_backend(), str) and matplotlib.get_backend()
assert np.finfo(float).eps < 3e-16
assert scipy.linalg.eigvals is eigvals

waktu_mulai_notebook = time.perf_counter()
print("Versi tervalidasi:", versi_aktual)
print("Backend Matplotlib:", matplotlib.get_backend())

## 1. Relasi dispersi dan penutupan Soal 1–3

Untuk gelombang bidang yang ditulis secara konsisten sebagai

\[
\psi(x,y,t)=\exp[i(\omega t+qx+py)],\qquad Q=q^2+p^2,
\]

persamaan bagian real dan imajiner memberi hasil berikut.

- **Soal 1, \(\mu=0\):** harus berlaku \(Q=\Omega\), sehingga solusi bervektor gelombang real hanya ada jika \(\Omega\geq0\). Frekuensinya \(\omega=\nu-\eta\Omega\). Pada \(\Omega=0\), hanya \((q,p)=(0,0)\), jadi solusi unik. Untuk \(\Omega>0\), semua titik pada lingkaran \(q^2+p^2=\Omega\) adalah solusi, jadi solusi tidak unik.
- **Soal 2, \(\mu\ne0\) tanpa faktor pertumbuhan:** harus berlaku \(\mu>0\). Dengan \(r=\sqrt{\mu/\alpha}\), cabang radialnya \(Q=\Omega\pm r\), setelah cabang negatif dibuang. Jika \(\Omega<-r\), tidak ada solusi. Pada \(\Omega=-r\), satu-satunya cabang ialah \(Q=0\), sehingga hanya \((q,p)=(0,0)\) dan solusinya unik. Jika \(-r<\Omega<r\), terdapat satu lingkaran dengan \(Q>0\); jika \(\Omega\geq r\), sedikitnya satu cabang positif ada (pada \(\Omega=r\), bersama cabang \(Q=0\)). Karena setiap \(Q>0\) memberi seluruh lingkaran \(q^2+p^2=Q\), semua kasus tersebut tidak unik. Pada setiap cabang sah, \(\omega=\nu-\beta\mu/\alpha-\eta Q\). Jika \(\mu<0\), tidak ada solusi berbentuk yang diminta.
- **Soal 3:** untuk sembarang \(q,p\in\mathbb R\), eksponen real dan frekuensi adalah \(\lambda=\mu-\alpha(\Omega-Q)^2\) dan \(\omega=\nu-\beta(\Omega-Q)^2-\eta Q\). Keluarga ini tidak unik.

Kurva di bawah memakai \(\alpha=1\), \(\mu=0{,}25\), dan \(\Omega=\pm1\). Nilai-nilai itu hanya contoh yang memenuhi keterangan Gambar 10.3.

In [ ]:
def lambda_swift(k_kuadrat, mu, nu, alpha, beta, Omega, eta):
    k_kuadrat = np.asarray(k_kuadrat, dtype=float)
    return (
        mu
        + 1j * nu
        - (alpha + 1j * beta) * (Omega - k_kuadrat) ** 2
        - 1j * eta * k_kuadrat
    )

def sigma_swift(k, mu, alpha, Omega):
    k = np.asarray(k, dtype=float)
    return mu - alpha * (Omega - k * k) ** 2

def cabang_Q_soal_1(Omega):
    return np.array([Omega], dtype=float) if Omega >= 0.0 else np.empty(0, dtype=float)

def cabang_Q_soal_2(mu, alpha, Omega):
    assert alpha > 0.0
    if mu < 0.0:
        return np.empty(0, dtype=float)
    akar = np.sqrt(mu / alpha)
    kandidat = np.array([Omega - akar, Omega + akar], dtype=float)
    return np.unique(np.round(kandidat[kandidat >= 0.0], 14))

alpha_demo = 1.0
mu_demo = 0.25
k_kurva = np.linspace(0.0, 2.2, 2201)
sigma_negatif = sigma_swift(k_kurva, mu_demo, alpha_demo, -1.0)
sigma_positif = sigma_swift(k_kurva, mu_demo, alpha_demo, 1.0)
indeks_negatif = int(np.argmax(sigma_negatif))
indeks_positif = int(np.argmax(sigma_positif))
k_c = 1.0
l_c = 2.0 * np.pi / k_c
batas_bawah = np.sqrt(1.0 - np.sqrt(mu_demo / alpha_demo))
batas_atas = np.sqrt(1.0 + np.sqrt(mu_demo / alpha_demo))

assert k_kurva.shape == sigma_negatif.shape == sigma_positif.shape
assert np.all(np.isfinite(sigma_negatif)) and np.all(np.isfinite(sigma_positif))
assert indeks_negatif == 0 and k_kurva[indeks_negatif] == 0.0
assert np.isclose(sigma_negatif[indeks_negatif], -0.75)
assert np.isclose(k_kurva[indeks_positif], k_c, atol=1e-12)
assert np.isclose(sigma_positif[indeks_positif], mu_demo, atol=1e-12)
assert np.isclose(l_c, 2.0 * np.pi)
assert np.isclose(sigma_swift(batas_bawah, mu_demo, alpha_demo, 1.0), 0.0)
assert np.isclose(sigma_swift(batas_atas, mu_demo, alpha_demo, 1.0), 0.0)
assert batas_bawah < k_c < batas_atas

parameter_uji = dict(mu=0.37, nu=0.41, alpha=1.2, beta=-0.23, Omega=0.8, eta=0.17)
Q_uji = np.array([0.0, 0.2, 0.8, 1.4])
lambda_uji = lambda_swift(Q_uji, **parameter_uji)
assert np.allclose(
    lambda_uji.real,
    parameter_uji["mu"] - parameter_uji["alpha"] * (parameter_uji["Omega"] - Q_uji) ** 2,
)
assert np.allclose(
    lambda_uji.imag,
    parameter_uji["nu"]
    - parameter_uji["beta"] * (parameter_uji["Omega"] - Q_uji) ** 2
    - parameter_uji["eta"] * Q_uji,
)

sudut = np.linspace(0.0, 2.0 * np.pi, 13, endpoint=False)
assert cabang_Q_soal_1(-0.1).size == 0
assert np.array_equal(cabang_Q_soal_1(0.0), [0.0])
assert np.array_equal(cabang_Q_soal_1(0.8), [0.8])
q_soal_1 = np.sqrt(0.8) * np.cos(sudut)
p_soal_1 = np.sqrt(0.8) * np.sin(sudut)
omega_soal_1 = parameter_uji["nu"] - parameter_uji["eta"] * parameter_uji["Omega"]
assert np.allclose(q_soal_1**2 + p_soal_1**2, parameter_uji["Omega"])
assert np.unique(np.round(q_soal_1, 12)).size > 2
assert np.isclose(omega_soal_1, 0.274)
assert cabang_Q_soal_2(-0.1, 1.0, 1.0).size == 0
assert cabang_Q_soal_2(0.25, 1.0, -1.0).size == 0
assert cabang_Q_soal_2(0.25, 1.0, -0.6).size == 0
assert np.array_equal(cabang_Q_soal_2(0.25, 1.0, -0.5), [0.0])
assert np.array_equal(cabang_Q_soal_2(0.25, 1.0, 0.0), [0.5])
assert np.array_equal(cabang_Q_soal_2(0.25, 1.0, 0.5), [0.0, 1.0])
assert np.allclose(cabang_Q_soal_2(0.25, 1.0, 1.0), [0.5, 1.5])
assert np.allclose(cabang_Q_soal_2(1.0, 1.0, -0.25), [0.75])

Q_cabang = cabang_Q_soal_2(0.25, 1.0, 1.0)
omega_cabang = 0.4 - (-0.2) * 0.25 - 0.1 * Q_cabang
lambda_cabang = lambda_swift(Q_cabang, 0.25, 0.4, 1.0, -0.2, 1.0, 0.1)
assert np.allclose(lambda_cabang.real, 0.0, atol=2e-14)
assert np.allclose(lambda_cabang.imag, omega_cabang)

Q_soal_3 = 0.73
lambda_soal_3 = lambda_swift(Q_soal_3, **parameter_uji)
assert np.isclose(lambda_soal_3.real, parameter_uji["mu"] - parameter_uji["alpha"] * (parameter_uji["Omega"] - Q_soal_3) ** 2)
assert np.isclose(lambda_soal_3.imag, parameter_uji["nu"] - parameter_uji["beta"] * (parameter_uji["Omega"] - Q_soal_3) ** 2 - parameter_uji["eta"] * Q_soal_3)

panjang_kotak_uji = 23.0
indeks_kotak = np.arange(-20, 21)
kx_kotak, ky_kotak = np.meshgrid(
    2.0 * np.pi * indeks_kotak / panjang_kotak_uji,
    2.0 * np.pi * indeks_kotak / panjang_kotak_uji,
    indexing="xy",
)
k2_kotak = kx_kotak**2 + ky_kotak**2
mu_kritis_kotak = float(np.min(alpha_demo * (1.0 - k2_kotak) ** 2))
indeks_terdekat = int(np.argmin(np.abs(k2_kotak - 1.0)))
k_terdekat = float(np.sqrt(k2_kotak.ravel()[indeks_terdekat]))
assert 0.0 < mu_kritis_kotak < 1e-3
assert np.isclose(mu_kritis_kotak, alpha_demo * (1.0 - k_terdekat**2) ** 2)
assert abs(k_terdekat - 1.0) < 2.0 * np.pi / panjang_kotak_uji

fig_dispersi, ax_dispersi = plt.subplots(figsize=(8.8, 5.0), constrained_layout=True)
ax_dispersi.plot(k_kurva, sigma_negatif, color="#D55E00", label=r"$\Omega=-1$")
ax_dispersi.plot(k_kurva, sigma_positif, color="#0072B2", label=r"$\Omega=1$")
ax_dispersi.axhline(0.0, color="#222222", linewidth=0.9)
ax_dispersi.axvline(k_c, color="#009E73", linestyle="--", label=r"$k_c=1$")
ax_dispersi.set(
    xlabel="bilangan gelombang k",
    ylabel=r"laju pertumbuhan $\sigma(k)$",
    title="Kurva pertumbuhan representatif, bukan reproduksi piksel",
)
ax_dispersi.grid(alpha=0.2)
ax_dispersi.legend()
akhiri_gambar(fig_dispersi)

print(f"Kasus Omega>0: kc={k_c:.6f}, lc={l_c:.6f}, pita=({batas_bawah:.6f}, {batas_atas:.6f})")
print(f"Kasus Omega<0: maksimum sigma={sigma_negatif[indeks_negatif]:.6f} pada k=0")
print(f"Kotak uji L={panjang_kotak_uji:g}: k terdekat={k_terdekat:.9f}, mu kritis={mu_kritis_kotak:.9e}")

## 2. Simulasi spektral dua dimensi yang terbuka

Untuk koefisien yang dinyatakan di bawah, operator linear diterapkan tepat di ruang Fourier selama setiap langkah, sedangkan nonlinieritas dievaluasi di ruang fisik. Pembaruan ETD orde satu ialah

\[
\widehat\psi_{n+1}=e^{\Delta tL}\widehat\psi_n
+\Delta t\,\varphi_1(\Delta tL)\,\widehat{N(\psi_n)},
\qquad \varphi_1(z)=\frac{e^z-1}{z}.
\]

Kasus stasioner memakai semua bagian imajiner nol dan \(\zeta=0\). Karena subruang keadaan real invariant secara eksak, bagian real diproyeksikan kembali setelah setiap transformasi balik untuk mencegah derau pembulatan kompleks ikut tumbuh. Kondisi awal mengandung modus \(k=1\) dan gangguan acak kecil yang berbiji tetap.

Sebagai demonstrasi kasus bergantung waktu yang dapat diuji tanpa pencarian parameter, ambil hanya \(\nu\ne0\). Jika \(u_*(x,y)\) merupakan pola stasioner dari sistem real, maka \(\psi(x,y,t)=e^{i\nu t}u_*(x,y)\) merupakan solusi eksak sistem kompleks ketika \(\zeta=0\). Intensitasnya tetap, tetapi bagian realnya berubah terhadap waktu. Demonstrasi ini tidak menyatakan adanya perjalanan spasial.

In [ ]:
BIJI_RNG = 19420260822
N_KISI = 64
PANJANG = 16.0 * np.pi
DX = PANJANG / N_KISI
x = np.arange(N_KISI, dtype=float) * DX
X, Y = np.meshgrid(x, x, indexing="xy")
k_satu = 2.0 * np.pi * np.fft.fftfreq(N_KISI, d=DX)
KX, KY = np.meshgrid(k_satu, k_satu, indexing="xy")
K2 = KX * KX + KY * KY

PARAMETER_STASIONER = {
    "mu": 0.30,
    "nu": 0.0,
    "alpha": 1.0,
    "beta": 0.0,
    "Omega": 1.0,
    "eta": 0.0,
    "gamma": 1.0,
    "delta": 0.0,
    "zeta": 0.0,
}

def operator_linear(K2_lokal, parameter):
    return lambda_swift(
        K2_lokal,
        parameter["mu"],
        parameter["nu"],
        parameter["alpha"],
        parameter["beta"],
        parameter["Omega"],
        parameter["eta"],
    )

def nonlinier_swift(psi, parameter):
    return (
        -(parameter["gamma"] + 1j * parameter["delta"]) * np.abs(psi) ** 2 * psi
        - parameter["zeta"] * np.abs(psi) ** 2
    )

def pengali_etd(dt, parameter):
    L = operator_linear(K2, parameter)
    z = dt * L
    E = np.exp(z)
    phi_1 = np.ones_like(z, dtype=complex)
    mask = np.abs(z) > 1e-12
    phi_1[mask] = np.expm1(z[mask]) / z[mask]
    assert np.all(np.isfinite(E)) and np.all(np.isfinite(phi_1))
    return L, E, phi_1

def simulasi_swift(awal, dt, jumlah_langkah, parameter, paksa_real=False, interval_cuplikan=None):
    assert dt > 0.0 and isinstance(jumlah_langkah, int) and jumlah_langkah > 0
    L, E, phi_1 = pengali_etd(dt, parameter)
    psi = np.asarray(awal, dtype=complex).copy()
    assert psi.shape == K2.shape
    cuplikan = []
    for langkah in range(jumlah_langkah):
        spektrum_baru = (
            E * np.fft.fft2(psi)
            + dt * phi_1 * np.fft.fft2(nonlinier_swift(psi, parameter))
        )
        psi = np.fft.ifft2(spektrum_baru)
        if paksa_real:
            psi = psi.real.astype(complex)
        if interval_cuplikan and (langkah + 1) % interval_cuplikan == 0:
            cuplikan.append(psi.copy())
    assert np.all(np.isfinite(psi.real)) and np.all(np.isfinite(psi.imag))
    return psi, cuplikan

assert N_KISI == 64 and K2.shape == (64, 64)
assert np.isclose(DX, np.pi / 4.0)
assert np.isclose(k_satu[1] - k_satu[0], 0.125)
assert np.isclose(np.sqrt(K2[0, 8]), 1.0)

rng_1 = np.random.default_rng(BIJI_RNG)
rng_2 = np.random.default_rng(BIJI_RNG)
derau_1 = rng_1.standard_normal((N_KISI, N_KISI))
derau_2 = rng_2.standard_normal((N_KISI, N_KISI))
assert np.array_equal(derau_1, derau_2)
awal_pola = 0.05 * np.cos(X) + 0.001 * derau_1
assert awal_pola.shape == (N_KISI, N_KISI)
assert np.all(np.isfinite(awal_pola))

ulang_1, _ = simulasi_swift(awal_pola, 0.2, 20, PARAMETER_STASIONER, paksa_real=True)
ulang_2, _ = simulasi_swift(awal_pola, 0.2, 20, PARAMETER_STASIONER, paksa_real=True)
assert np.array_equal(ulang_1, ulang_2)

awal_modus = 1e-8 * np.cos(X)
waktu_awal = 0.8
hasil_modus, _ = simulasi_swift(
    awal_modus,
    0.1,
    int(round(waktu_awal / 0.1)),
    PARAMETER_STASIONER,
    paksa_real=True,
)
amplitudo_awal = abs(np.fft.fft2(awal_modus)[0, 8])
amplitudo_akhir = abs(np.fft.fft2(hasil_modus)[0, 8])
rasio_pertumbuhan = float(amplitudo_akhir / amplitudo_awal)
rasio_teori = float(np.exp(PARAMETER_STASIONER["mu"] * waktu_awal))
assert amplitudo_awal > 0.0 and amplitudo_akhir > amplitudo_awal
assert abs(rasio_pertumbuhan / rasio_teori - 1.0) < 1e-10

pola_stasioner, cuplikan = simulasi_swift(
    awal_pola,
    0.2,
    600,
    PARAMETER_STASIONER,
    paksa_real=True,
    interval_cuplikan=100,
)
assert len(cuplikan) == 6
assert np.max(np.abs(pola_stasioner.imag)) == 0.0
assert np.max(np.abs(pola_stasioner.real)) < 1.0
perubahan_akhir = float(np.linalg.norm(cuplikan[-1] - cuplikan[-2]) / np.linalg.norm(cuplikan[-1]))
assert perubahan_akhir < 1e-4

L_stasioner = operator_linear(K2, PARAMETER_STASIONER)
residu_stasioner = (
    np.fft.ifft2(L_stasioner * np.fft.fft2(pola_stasioner))
    + nonlinier_swift(pola_stasioner, PARAMETER_STASIONER)
)
residu_relatif = float(np.linalg.norm(residu_stasioner) / np.linalg.norm(pola_stasioner))
assert residu_relatif < 5e-6

daya = np.abs(np.fft.fft2(pola_stasioner)) ** 2
daya[0, 0] = 0.0
indeks_dominan = int(np.argmax(daya))
k_dominan = float(np.sqrt(K2.ravel()[indeks_dominan]))
delta_k = 2.0 * np.pi / PANJANG
assert np.max(daya) > 0.0
assert abs(k_dominan - np.sqrt(PARAMETER_STASIONER["Omega"])) <= delta_k
assert np.isclose(k_dominan, 1.0, atol=1e-12)

kasar, _ = simulasi_swift(awal_pola, 0.2, 50, PARAMETER_STASIONER, paksa_real=True)
halus, _ = simulasi_swift(awal_pola, 0.1, 100, PARAMETER_STASIONER, paksa_real=True)
galat_penghalusan = float(np.linalg.norm(kasar - halus) / np.linalg.norm(halus))
assert 0.0 < galat_penghalusan < 0.02

nu_dinamis = 0.4
selang_fase = np.pi / nu_dinamis
pola_t0 = pola_stasioner.astype(complex)
pola_t1 = np.exp(1j * nu_dinamis * selang_fase) * pola_t0
assert np.allclose(np.abs(pola_t1), np.abs(pola_t0), atol=2e-14)
assert np.linalg.norm(pola_t1.real - pola_t0.real) / np.linalg.norm(pola_t0.real) > 1.99
mask_tak_nol = np.abs(pola_t0) > 1e-8
assert np.allclose(pola_t1[mask_tak_nol] / pola_t0[mask_tak_nol], -1.0 + 0.0j, atol=2e-14)

fig_pola, sumbu_pola = plt.subplots(2, 2, figsize=(10.2, 8.7), constrained_layout=True)
gambar_0 = sumbu_pola[0, 0].imshow(awal_pola, cmap="RdBu_r", origin="lower")
sumbu_pola[0, 0].set_title("Kondisi awal berbiji tetap")
gambar_1 = sumbu_pola[0, 1].imshow(pola_t0.real, cmap="RdBu_r", origin="lower")
sumbu_pola[0, 1].set_title("Pola stasioner, bagian real")
gambar_2 = sumbu_pola[1, 0].imshow(pola_t1.real, cmap="RdBu_r", origin="lower")
sumbu_pola[1, 0].set_title(r"Kasus $\nu\ne0$ setelah rotasi fase $\pi$")
spektrum_tampil = np.fft.fftshift(np.log10(1.0 + daya))
gambar_3 = sumbu_pola[1, 1].imshow(spektrum_tampil, cmap="viridis", origin="lower")
sumbu_pola[1, 1].set_title("Spektrum daya logaritmik")
for ax in sumbu_pola.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
for ax, gambar in zip(sumbu_pola.ravel(), (gambar_0, gambar_1, gambar_2, gambar_3)):
    fig_pola.colorbar(gambar, ax=ax, shrink=0.76)
akhiri_gambar(fig_pola)

print(f"Pertumbuhan modus awal: numerik={rasio_pertumbuhan:.12f}, teori={rasio_teori:.12f}")
print(f"k dominan={k_dominan:.9f}; toleransi satu bin={delta_k:.9f}")
print(f"Perubahan cuplikan akhir={perubahan_akhir:.9e}; residu={residu_relatif:.9e}")
print(f"Galat penghalusan dt 0.2 terhadap 0.1 pada t=10: {galat_penghalusan:.9e}")

## 3. Penutupan terbuka model vegetasi Klausmeier

Soal 4–5 merujuk persamaan dimensional (1) dalam [Klausmeier (1999)](https://doi.org/10.1126/science.284.5421.1826), tetapi bab dan EPUB beku tidak membundel artikel itu. Agar aljabar dapat diperiksa secara mandiri, persamaan dimensional tersebut ditranskripsikan dan dinyatakan ulang dari artikel primer yang dikutip; tidak ada berkas atau kode artikel yang dibundel:

\[
W_T=A-LW-RWN^2+VW_X,
\qquad
N_T=JRWN^2-MN+D(N_{XX}+N_{YY}).
\]

Dengan \(T_0=L^{-1}\), \(X_0=\sqrt{D/L}\), \(N_0=\sqrt{L/R}\), dan \(W_0=N_0/J\), definisikan

\[
t=LT,\quad (x,y)=\sqrt{L/D}(X,Y),\quad
n=N\sqrt{R/L},\quad w=JW\sqrt{R/L},
\]
\[
a=\frac{AJ\sqrt R}{L^{3/2}},\qquad m=\frac ML,
\qquad v=\frac V{\sqrt{LD}}.
\]

Secara fisik, \(W\) menyatakan massa air per satuan luas (misalnya kg H₂O m⁻²), sedangkan \(N\) menyatakan massa kering biomassa tumbuhan per satuan luas (misalnya kg massa kering m⁻²). Dimensi formalnya adalah \([A]=[W]/T\), \([L]=[M]=T^{-1}\), \([R]=[N]^{-2}T^{-1}\), \([V]=\text{panjang}/T\), \([J]=[N]/[W]\), dan \([D]=\text{panjang}^2/T\). Substitusi menghasilkan sistem tak berdimensi yang dicetak dalam bab.

Titik tanpa vegetasi adalah \((w,n)=(a,0)\). Untuk \(a\geq2m\), titik-titik vegetasi memenuhi

\[
n_\pm=\frac{a\pm\sqrt{a^2-4m^2}}{2m},\qquad
w_\pm=\frac{m}{n_\pm}=\frac{a\mp\sqrt{a^2-4m^2}}2.
\]

Kedua titik vegetasi berbeda jika \(a>2m\); pada \(a=2m\), keduanya berimpit menjadi satu titik pelana-simpul ganda \((w,n)=(m,1)\).

Jacobian reaksinya ialah

\[
J(w,n)=\begin{pmatrix}-(1+n^2)&-2wn\\n^2&2wn-m\end{pmatrix}.
\]

Di titik vegetasi, \(wn=m\). Untuk gangguan \(e^{\lambda t+i(k_xx+k_yy)}\), matriks Fourier dan determinannya adalah

\[
A(\mathbf k)=\begin{pmatrix}-1-n^2+ivk_x&-2m\\n^2&m-k^2\end{pmatrix},
\]
\[
\det A=m(n^2-1)+(1+n^2)k^2+ivk_x(m-k^2).
\]

Contoh numerik \(a=0{,}95\), \(m=0{,}45\), \(v=20\) dipilih secara mandiri untuk menguji rumus dan menunjukkan ketakstabilan pada bilangan gelombang hingga sepanjang arah lereng. Contoh ini bukan parameter Gambar 2 artikel dan tidak diklaim mereproduksi gambar tersebut.

In [ ]:
def reaksi_klausmeier(w, n, a, m):
    return np.array([a - w - w * n * n, w * n * n - m * n], dtype=float)

def titik_klausmeier(a, m):
    assert a > 0.0 and m > 0.0
    titik = [(a, 0.0)]
    diskriminan_kuadrat = a * a - 4.0 * m * m
    if diskriminan_kuadrat == 0.0:
        titik.append((m, 1.0))
    elif diskriminan_kuadrat > 0.0:
        diskriminan = np.sqrt(diskriminan_kuadrat)
        for tanda in (-1.0, 1.0):
            n = (a + tanda * diskriminan) / (2.0 * m)
            w = m / n
            titik.append((w, n))
    return titik

def jacobian_reaksi(w, n, m):
    return np.array(
        [[-(1.0 + n * n), -2.0 * w * n], [n * n, 2.0 * w * n - m]],
        dtype=float,
    )

def matriks_fourier_vegetasi(n, m, v, kx, ky):
    k2 = kx * kx + ky * ky
    return np.array(
        [[-1.0 - n * n + 1j * v * kx, -2.0 * m], [n * n, m - k2]],
        dtype=complex,
    )

A_dim, L_dim, R_dim = 1.7, 0.8, 0.6
V_dim, J_dim, M_dim, D_dim = 2.3, 1.4, 0.22, 0.9
T0 = 1.0 / L_dim
X0 = np.sqrt(D_dim / L_dim)
N0 = np.sqrt(L_dim / R_dim)
W0 = N0 / J_dim
a_dimless = A_dim * J_dim * np.sqrt(R_dim) / L_dim**1.5
m_dimless = M_dim / L_dim
v_dimless = V_dim / np.sqrt(L_dim * D_dim)

assert np.isclose(L_dim * T0, 1.0)
assert np.isclose(D_dim / (L_dim * X0**2), 1.0)
assert np.isclose(R_dim * N0**2 / L_dim, 1.0)
assert np.isclose(J_dim * R_dim * W0 * N0 / L_dim, 1.0)
assert np.isclose(A_dim / (L_dim * W0), a_dimless)
assert np.isclose(M_dim / L_dim, m_dimless)
assert np.isclose(V_dim / (L_dim * X0), v_dimless)
assert np.all(np.array([T0, X0, N0, W0, a_dimless, m_dimless, v_dimless]) > 0.0)

a_veg, m_veg, v_veg = 0.95, 0.45, 20.0
titik = titik_klausmeier(a_veg, m_veg)
assert titik_klausmeier(0.9, 0.45) == [(0.9, 0.0), (0.45, 1.0)]
assert len(titik) == 3
assert np.allclose(titik[0], [a_veg, 0.0])
for w, n in titik:
    assert np.linalg.norm(reaksi_klausmeier(w, n, a_veg, m_veg)) < 2e-14
    assert np.all(np.isfinite(jacobian_reaksi(w, n, m_veg)))

(w_nir, n_nir), (w_minus, n_minus), (w_plus, n_plus) = titik
assert n_minus < 1.0 < n_plus
assert np.isclose(n_minus * n_plus, 1.0)
assert np.isclose(w_minus * n_minus, m_veg)
assert np.isclose(w_plus * n_plus, m_veg)
assert np.isclose(w_minus + w_plus, a_veg)

J_nir = jacobian_reaksi(w_nir, n_nir, m_veg)
J_plus = jacobian_reaksi(w_plus, n_plus, m_veg)
assert np.allclose(J_nir, [[-1.0, 0.0], [0.0, -m_veg]])
assert np.allclose(J_plus, [[-1.0 - n_plus**2, -2.0 * m_veg], [n_plus**2, m_veg]])
assert np.max(eigvals(J_nir).real) < 0.0
assert np.max(eigvals(J_plus).real) < 0.0
assert np.linalg.det(J_plus) > 0.0 and np.trace(J_plus) < 0.0

kx_uji, ky_uji = 0.31, 0.17
k2_uji = kx_uji**2 + ky_uji**2
A_uji = matriks_fourier_vegetasi(n_plus, m_veg, v_veg, kx_uji, ky_uji)
jejak_rumus = m_veg - 1.0 - n_plus**2 - k2_uji + 1j * v_veg * kx_uji
determinan_rumus = (
    m_veg * (n_plus**2 - 1.0)
    + (1.0 + n_plus**2) * k2_uji
    + 1j * v_veg * kx_uji * (m_veg - k2_uji)
)
assert np.isclose(np.trace(A_uji), jejak_rumus)
assert np.isclose(np.linalg.det(A_uji), determinan_rumus)
nilai_eigen_uji = eigvals(A_uji)
assert np.isclose(np.sum(nilai_eigen_uji), jejak_rumus)
assert np.isclose(np.prod(nilai_eigen_uji), determinan_rumus)

k_dispersi = np.linspace(0.0, 1.0, 2001)
pertumbuhan_lereng = np.array([
    np.max(eigvals(matriks_fourier_vegetasi(n_plus, m_veg, v_veg, k, 0.0)).real)
    for k in k_dispersi
])
pertumbuhan_melintang = np.array([
    np.max(eigvals(matriks_fourier_vegetasi(n_plus, m_veg, v_veg, 0.0, k)).real)
    for k in k_dispersi
])
indeks_puncak_lereng = int(np.argmax(pertumbuhan_lereng))
k_puncak_lereng = float(k_dispersi[indeks_puncak_lereng])
assert np.all(np.isfinite(pertumbuhan_lereng)) and np.all(np.isfinite(pertumbuhan_melintang))
assert pertumbuhan_lereng[0] < 0.0
assert np.max(pertumbuhan_lereng) > 0.20
assert 0.28 < k_puncak_lereng < 0.31
assert np.max(pertumbuhan_melintang) < 0.0
assert int(np.argmax(pertumbuhan_melintang)) == 0

fig_klaus, ax_klaus = plt.subplots(figsize=(8.8, 5.0), constrained_layout=True)
ax_klaus.plot(k_dispersi, pertumbuhan_lereng, color="#0072B2", label="sepanjang lereng (ky=0)")
ax_klaus.plot(k_dispersi, pertumbuhan_melintang, color="#D55E00", label="melintang (kx=0)")
ax_klaus.axhline(0.0, color="#222222", linewidth=0.9)
ax_klaus.axvline(k_puncak_lereng, color="#009E73", linestyle="--", label="puncak contoh mandiri")
ax_klaus.set(
    xlabel="bilangan gelombang k",
    ylabel="bagian real nilai eigen terbesar",
    title="Dispersi Klausmeier untuk parameter demonstrasi yang dinyatakan",
)
ax_klaus.grid(alpha=0.2)
ax_klaus.legend()
akhiri_gambar(fig_klaus)

waktu_notebook = time.perf_counter() - waktu_mulai_notebook
assert waktu_notebook < 120.0

print("Titik tetap Klausmeier (w, n):", titik)
print(f"Pertumbuhan homogen cabang atas={pertumbuhan_lereng[0]:.9f}")
print(f"Puncak arah lereng={np.max(pertumbuhan_lereng):.9f} pada k={k_puncak_lereng:.6f}")
print(f"Waktu eksekusi sel komputasi={waktu_notebook:.3f} s")

## Kesimpulan

Relasi dispersi Swift–Hohenberg, ambang kontinu, koreksi ambang pada kotak hingga, dan semua cabang keberadaan Soal 1–3 telah diperiksa secara numerik terhadap rumus analitik. Simulasi dua dimensi yang sederhana mempertahankan keluaran hingga, mengulang hasil secara deterministik, mereproduksi laju awal modus \(k_c\), memilih bilangan gelombang dominan dalam satu bin Fourier, dan lulus pemeriksaan penghalusan langkah waktu.

Kasus dengan koefisien real mendekati pola stasioner. Menambahkan hanya bagian imajiner \(i\nu\psi\) memberi rotasi fase eksak: intensitas tetap sedangkan bagian real berubah terhadap waktu. Ini menunjukkan perbedaan yang dinyatakan bab tanpa mengklaim telah mereproduksi dinamika lengkap atau piksel Gambar 10.2.

Untuk model Klausmeier, notebook menjadikan pendahulu dimensional, penskalaan, titik tetap, Jacobian, dan determinan Fourier dapat diperiksa tanpa kode berpemilik. Parameter demonstrasi dipilih secara mandiri dan hanya menguji mekanisme; reproduksi Gambar 2 artikel tetap memerlukan parameter dan konteks asli yang tidak dibundel dalam sumber beku.